# From CSV to Risk Classification — a Walkthrough

**The situation.** An insurance carrier (or lender, or property-tech
platform) has a CSV of properties dropped in their inbox every morning:
street address + latitude/longitude. They need each row classified as
`MINIMAL / MODERATE / HIGH / EXTREME` flood risk before it hits
underwriting review — with an auditable trail back to the FEMA zone it
came from.

This notebook walks the full happy path in six steps:

| # | Step | Wherobots primitive |
|---|---|---|
| 1 | Read the CSV | `sedona.read.csv` |
| 2 | Build point geometries from lat/lon | `ST_SetSRID(ST_Point(...), 4326)` |
| 3 | Geocode to admin context (city / county / state) | Spatial join to Regrid parcels |
| 4 | Look up FEMA flood zone | Same join — parcels carry FEMA designations |
| 5 | Classify each property into a risk tier | Spark SQL `CASE` expression |
| 6 | Ship results — CSV + GeoJSON | `toPandas().to_csv`, `ST_AsGeoJSON` |

The goal is to show how few moving parts this is on Wherobots — the
entire pipeline is one notebook, no ML team, no GIS specialist required.

## 1. Setup

One Sedona context, nothing else. The rest of the notebook is just SQL
and DataFrame operations.

In [ ]:
from sedona.spark import *
import pyspark.sql.functions as f
import json

config = SedonaContext.builder().getOrCreate()
sedona = SedonaContext.create(config)

## 2. The Input — a Raw Property CSV

For the demo we write a representative CSV to `/tmp` first — in
production this is the file dropped in an SFTP / S3 / SharePoint location
every morning. Sample rows span coastal California so we exercise the
full spread of FEMA zones (VE, AE, X-levee, X-500yr).

In [ ]:
raw_csv_path = "/tmp/underwriting_queue.csv"

csv_text = """property_id,street_address,city_given,state_given,lat,lon,insured_value
UW-0001,100 Ocean Pier Way,Huntington Beach,CA,33.6553,-117.9988,1850000
UW-0002,812 Pacific Cove,Sunset Beach,CA,33.7192,-118.0709,1250000
UW-0003,441 Balboa Blvd,Newport Beach,CA,33.6019,-117.9022,3400000
UW-0004,2730 E 2nd St,Long Beach,CA,33.7589,-118.1274,980000
UW-0005,55 Ocean Dr,Newport Beach,CA,33.5950,-117.8740,4200000
UW-0006,301 Marina Dr,Seal Beach,CA,33.7410,-118.1058,875000
UW-0007,17 Harbour Ln,Huntington Harbour,CA,33.7233,-118.0690,2150000
UW-0008,920 Bristol St,Costa Mesa,CA,33.6411,-117.9186,740000
UW-0009,1201 Strand,Manhattan Beach,CA,33.8847,-118.4109,3100000
UW-0010,88 Washington Blvd,Marina del Rey,CA,33.9770,-118.4481,1650000
UW-0011,3400 Ocean Front,Venice,CA,33.9862,-118.4757,2900000
UW-0012,450 Palisades Ave,Santa Monica,CA,34.0194,-118.5062,5400000
UW-0013,710 Alameda Ave,Alameda,CA,37.7799,-122.2822,1100000
UW-0014,2100 Marina Bay Dr,Richmond,CA,37.9106,-122.3637,925000
UW-0015,1350 Beach Ave,Half Moon Bay,CA,37.4636,-122.4286,1480000
"""
with open(raw_csv_path, "w") as fh:
    fh.write(csv_text)

print(f"Sample underwriting queue: {raw_csv_path}")
print(f"  {len(csv_text.strip().splitlines()) - 1} rows")

## 3. Load & Build Point Geometries

`sedona.read.csv` reads like any Spark CSV; `ST_SetSRID(ST_Point(...))`
turns the lat/lon into a proper geometry. Everything downstream (joins,
distance, map export) operates on that column.

In [ ]:
raw_df = sedona.read.csv(raw_csv_path, header=True, inferSchema=True)

properties_df = raw_df.withColumn(
    "geometry",
    f.expr("ST_SetSRID(ST_Point(lon, lat), 4326)")
).cache()
properties_df.createOrReplaceTempView("properties")

print(f"Loaded {properties_df.count()} properties")
properties_df.printSchema()
properties_df.select("property_id", "city_given", "insured_value", "geometry") \
             .show(5, truncate=False)

## 4. Geocode — One Spatial Join Does Two Jobs

The Regrid parcel table (`wherobots_open_data.partner_samples.regrid_parcels`)
is an authoritative U.S. parcel layer with **FEMA flood zone embedded on
every polygon**. Intersecting our points with that layer simultaneously:

- **Reverse-geocodes** the point to its parcel → city, county, state,
  zip, use code, owner, tax info.
- **Looks up the FEMA flood zone** directly from the parcel attribute
  (no separate hazard join required).

Two tasks, one `ST_Intersects`.

In [ ]:
PARCEL_TABLE = "wherobots_open_data.partner_samples.regrid_parcels"

geocoded_df = sedona.sql(f"""
    SELECT
        p.property_id,
        p.street_address,
        p.city_given,
        p.state_given,
        p.lat,
        p.lon,
        p.insured_value,
        p.geometry,
        r.scity                                   AS parcel_city,
        r.county                                  AS parcel_county,
        r.state2                                  AS parcel_state,
        r.szip5                                   AS parcel_zip,
        r.usedesc                                 AS parcel_use,
        COALESCE(r.fema_flood_zone, 'UNMAPPED')   AS fema_zone,
        COALESCE(r.fema_flood_zone_subtype, '')   AS fema_subtype
    FROM properties p
    LEFT JOIN {PARCEL_TABLE} r
      ON ST_Intersects(p.geometry, r.geometry)
""").cache()
geocoded_df.createOrReplaceTempView("geocoded")

geocoded_df.select("property_id", "city_given", "parcel_city",
                   "parcel_county", "parcel_zip", "fema_zone",
                   "fema_subtype") \
           .show(truncate=False)

### Sanity-check the geocode

Compare the `city_given` (what underwriting typed in) to `parcel_city`
(what the parcel actually resolves to). Mismatches are legitimate flags
— the record's coordinates disagree with its stated address, which in
production routes the file to a human reviewer.

In [ ]:
sedona.sql("""
    SELECT
        property_id, street_address,
        city_given, parcel_city,
        CASE
            WHEN parcel_city IS NULL THEN 'NO_MATCH (outside parcel layer)'
            WHEN UPPER(city_given) = UPPER(parcel_city) THEN 'OK'
            ELSE 'CITY_MISMATCH'
        END AS geocode_status
    FROM geocoded
    ORDER BY geocode_status, property_id
""").show(truncate=False)

## 5. Classify — FEMA Zone → Risk Tier

Map the raw FEMA designation to a carrier-friendly tier and attach a
numeric score for accumulation / PML workflows.

| FEMA zone | Tier | Score | Meaning |
|---|---|---|---|
| VE, V | EXTREME | 1.00 | Coastal high hazard + wave action |
| A, AE, AO, AH, A99 | HIGH | 0.70 | 1% annual chance (100-yr floodplain) |
| X (0.2 PCT subtype) | MODERATE | 0.30 | 0.2% annual chance (500-yr) |
| X (LEVEE subtype) | LEVEE_PROTECTED | 0.15 | Behind an accredited levee |
| X (other) | MINIMAL | 0.05 | Outside mapped hazard |
| D / UNMAPPED | UNDETERMINED | 0.40 | Not studied — review |

In [ ]:
classified_df = sedona.sql("""
    SELECT
        property_id, street_address,
        parcel_city, parcel_county, parcel_state, parcel_zip,
        insured_value,
        fema_zone, fema_subtype,
        CASE
            WHEN fema_zone IN ('VE','V')                              THEN 'EXTREME'
            WHEN fema_zone IN ('A','AE','AO','AH','A99')              THEN 'HIGH'
            WHEN fema_zone = 'X' AND fema_subtype LIKE '%0.2 PCT%'    THEN 'MODERATE'
            WHEN fema_zone = 'X' AND fema_subtype LIKE '%LEVEE%'      THEN 'LEVEE_PROTECTED'
            WHEN fema_zone = 'X'                                      THEN 'MINIMAL'
            WHEN fema_zone IN ('D', 'UNMAPPED')                       THEN 'UNDETERMINED'
            ELSE                                                           'UNDETERMINED'
        END AS risk_tier,
        CASE
            WHEN fema_zone IN ('VE','V')                              THEN 1.00
            WHEN fema_zone IN ('A','AE','AO','AH','A99')              THEN 0.70
            WHEN fema_zone = 'X' AND fema_subtype LIKE '%0.2 PCT%'    THEN 0.30
            WHEN fema_zone = 'X' AND fema_subtype LIKE '%LEVEE%'      THEN 0.15
            WHEN fema_zone = 'X'                                      THEN 0.05
            ELSE                                                           0.40
        END AS risk_score,
        geometry
    FROM geocoded
""").cache()
classified_df.createOrReplaceTempView("classified")

classified_df.select("property_id", "parcel_city", "fema_zone",
                     "risk_tier", "risk_score", "insured_value") \
             .orderBy(f.col("risk_score").desc()) \
             .show(truncate=False)

### Portfolio-level summary

The number that matters to the underwriting manager: how much insured
value sits in each tier.

In [ ]:
sedona.sql("""
    SELECT
        risk_tier,
        COUNT(*)                                 AS property_count,
        ROUND(SUM(insured_value), 0)             AS total_tiv,
        ROUND(SUM(insured_value * risk_score), 0) AS exposed_tiv,
        ROUND(AVG(insured_value), 0)             AS avg_tiv
    FROM classified
    GROUP BY risk_tier
    ORDER BY
        CASE risk_tier
            WHEN 'EXTREME'         THEN 1
            WHEN 'HIGH'            THEN 2
            WHEN 'MODERATE'        THEN 3
            WHEN 'LEVEE_PROTECTED' THEN 4
            WHEN 'MINIMAL'         THEN 5
            ELSE                        6
        END
""").show(truncate=False)

## 6. Ship the Results

Two artifacts the underwriting system consumes:
- `/tmp/underwriting_classified.csv` — one row per property, ready for
  the policy-admin system.
- `/tmp/underwriting_classified.geojson` — same rows with point geometry,
  ready for the map layer.

In [ ]:
# CSV output — drop geometry for the tabular consumer
csv_out = "/tmp/underwriting_classified.csv"
classified_df.drop("geometry") \
             .orderBy("property_id") \
             .toPandas() \
             .to_csv(csv_out, index=False)
print(f"Wrote {csv_out}")

# GeoJSON output — for the underwriting dashboard map
geojson_rows = classified_df.selectExpr(
    "property_id", "street_address", "parcel_city", "parcel_state",
    "fema_zone", "risk_tier", "risk_score", "insured_value",
    "ST_AsGeoJSON(geometry) AS geojson"
).collect()

features = [
    {
        "type": "Feature",
        "properties": {
            "property_id":     r.property_id,
            "street_address":  r.street_address,
            "parcel_city":     r.parcel_city,
            "parcel_state":    r.parcel_state,
            "fema_zone":       r.fema_zone,
            "risk_tier":       r.risk_tier,
            "risk_score":      float(r.risk_score),
            "insured_value":   int(r.insured_value),
        },
        "geometry": json.loads(r.geojson),
    }
    for r in geojson_rows
]
geojson_out = "/tmp/underwriting_classified.geojson"
with open(geojson_out, "w") as fh:
    json.dump({"type": "FeatureCollection", "features": features}, fh, indent=2)
print(f"Wrote {geojson_out} ({len(features)} features)")

## 7. Preview on a Map

Visual sanity check — each point color-coded by its tier.

In [ ]:
SedonaKepler.create_map(
    classified_df.select("property_id", "parcel_city",
                         "fema_zone", "risk_tier", "risk_score",
                         "insured_value", "geometry"),
    name="Underwriting Queue — Risk Classification"
)

---

## What just happened

In ~100 lines of notebook, a raw CSV became:

- Reverse-geocoded against an authoritative parcel layer.
- Enriched with city / county / zip / use code from the underlying parcel.
- Tagged with FEMA flood zone.
- Classified into a carrier-friendly risk tier with a numeric score.
- Rolled up into portfolio exposure.
- Shipped to CSV + GeoJSON for downstream systems.

## Next steps

| Want to... | Replace |
|---|---|
| Handle 100k+ rows per run | Same code — switch runtime size; Spark handles it |
| Pull from SFTP / S3 / SharePoint | `sedona.read.csv("s3://bucket/queue/*.csv")` — glob works |
| Add wildfire + seismic perils | See `sa_catastrophe_exposure_model.ipynb` — same join pattern |
| Persist to a managed table | `classified_df.writeTo("org_catalog.uw.classified").createOrReplace()` |
| Refresh nightly | Wrap this notebook in a scheduled job or Airflow DAG |